# scANVI

In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import os
import sys
import pycats
from pandas.api.types import CategoricalDtype

scvi.settings.seed = 1234
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
adata = sc.read("Reproducibility/Data/External_cohort/adata_concat_w_external_cohort.h5ad")

In [ ]:
adata_TNK = adata[adata.obs['predicted_lineage'].isin(['TNK']) & adata.obs['coarse_celltype'].isin(['unknown', 'CD4_Tconv', 'CD8_T', 'Treg', 'NK_ILC'])
                  & adata.obs['FACS'].isin(['bulk','CD45pos'])].copy()

In [ ]:
def scANVI_integration(adata,tmp_lineage):
    sc.pp.highly_variable_genes(adata, 
                            n_top_genes=3000, 
                            flavor='seurat_v3',
                            layer="counts",
                            batch_key="paper",
                            subset=True)
    ###
    adata_DOGMA = adata[adata.obs['paper'].isin(['DOGMA'])==True].copy()
    adata_EXT = adata[adata.obs['paper'].isin(['DOGMA'])==False].copy()
    ###
    ## Integration with scVI ##
    scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="batch_id")
    scvi_model = scvi.model.SCVI(adata, n_layers=2, n_latent=30)
    ###
    scvi_model.train(max_epochs=1000, early_stopping=True)
    ###
    subset_path = f'Reproducibility/Results/scANVI/BC/{tmp_lineage}/'
    os.makedirs(subset_path, exist_ok = True)
    scvi_model_path = f'{subset_path}integration_batch_id/'
    scvi_model.save(scvi_model_path, overwrite=True)
    ###
    fig, ax = plt.subplots(1, 1)
    scvi_model.history["elbo_train"].plot(ax=ax, label="train")
    scvi_model.history["elbo_validation"].plot(ax=ax, label="validation")
    ax.set(title="Negative ELBO over training epochs")
    ax.legend()
    plt.savefig(f'{scvi_model_path}ELBO_curve_plot.pdf', bbox_inches='tight')
    plt.close()
    ###
    ## Load
    # scvi_model = scvi.model.SCVI.load(scvi_model_path, adata)
    SCVI_LATENT_KEY = "X_scVI"
    adata.obsm[SCVI_LATENT_KEY] = scvi_model.get_latent_representation()
    ###
    sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)
    sc.tl.umap(adata, min_dist=0.5)
    ###
    sc.set_figure_params(figsize=(3, 3))
    sc.set_figure_params(vector_friendly=True, dpi_save=100)
    sc.pl.umap(
        adata,
        color=["paper",'batch_id','celltype'],
        frameon=False,
        ncols=4,
    #    size=0.3,
        wspace = 0.5,
    #    legend_loc="on data",
        legend_fontsize=4,
        show = False
    )
    plt.savefig(f'{scvi_model_path}Atlas_level_integration_UMAP.pdf', bbox_inches='tight')
    plt.close()

In [ ]:
def scANVI_annotation_transfer(adata,tmp_lineage,celltype_use,categories_order):
    subset_path = f'Reproducibility/Results/scANVI/BC/{tmp_lineage}/'
    scvi_model_path = f'Reproducibility/Results/scANVI/BC/{tmp_lineage}/integration_batch_id/'
    scvi_model = scvi.model.SCVI.load(scvi_model_path, adata)
    ## Transfer of annotations with scANVI ##
    SCANVI_CELLTYPE_KEY = celltype_use
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        scvi_model,
        adata=adata,
        unlabeled_category="unknown",
        labels_key=SCANVI_CELLTYPE_KEY,
    )
    ###
    scanvi_model.train(max_epochs=1000, early_stopping=True, 
                       n_samples_per_label=adata.obs[SCANVI_CELLTYPE_KEY].value_counts().min())
    ###
    scanvi_model_path = f'{subset_path}SCANVI_{SCANVI_CELLTYPE_KEY}/'
    scanvi_model.save(scanvi_model_path, overwrite=True)
    ###
    fig, ax = plt.subplots(1, 1)
    scanvi_model.history["elbo_train"].plot(ax=ax, label="train")
    scanvi_model.history["elbo_validation"].plot(ax=ax, label="validation")
    ax.set(title="Negative ELBO over training epochs")
    plt.savefig(f'{scanvi_model_path}ELBO_curve_plot.pdf', bbox_inches='tight')
    plt.close()
    ###
    ## Load
    # scanvi_model = scvi.model.SCANVI.load(scanvi_model_path, adata)
    ## scanvae = sca.models.SCANVI.load(scanvi_model, adata)
    ###
    SCANVI_LATENT_KEY = "X_scANVI"
    SCANVI_PREDICTION_KEY = f'predicted_{SCANVI_CELLTYPE_KEY}'
    ###
    adata.obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation(adata)
    adata.obs[SCANVI_PREDICTION_KEY] = scanvi_model.predict(adata)
    ###
    sc.pp.neighbors(adata, use_rep=SCANVI_LATENT_KEY)
    sc.tl.umap(adata, min_dist=0.5)
    ###
    categories_order2 = list(filter(lambda x: x != 'unknown', categories_order))
    ###
    adata.obs[SCANVI_CELLTYPE_KEY] = pd.Categorical(adata.obs[SCANVI_CELLTYPE_KEY], categories=categories_order, ordered=True)
    adata.obs[SCANVI_PREDICTION_KEY] = pd.Categorical(adata.obs[SCANVI_PREDICTION_KEY], categories=categories_order2, ordered=True)
    ###
    sc.set_figure_params(figsize=(3, 3))
    sc.set_figure_params(vector_friendly=True, dpi_save=100)
    sc.pl.umap(
        adata,
        color=["paper",'batch_id','celltype',SCANVI_PREDICTION_KEY],
        frameon=False,
        ncols=4,
    #    size=0.3,
        wspace = 0.5,
    #    legend_loc="on data",
        legend_fontsize=4,
        show = False
    )
    plt.savefig(f'{subset_path}Transfer_annotation_SCANVI_UMAP.pdf', bbox_inches='tight')
    plt.close()
    ###
    # concordance
    tmp = adata[adata.obs['paper'].isin(['DOGMA'])==True].copy()
    ###
    accuracy = np.mean(tmp.obs[SCANVI_PREDICTION_KEY].astype(str) == tmp.obs[SCANVI_CELLTYPE_KEY].astype(str))
    print("Acc: {}".format(accuracy))
    ###
    df = tmp.obs.groupby([SCANVI_CELLTYPE_KEY, SCANVI_PREDICTION_KEY]).size().unstack(fill_value=0)
    norm_df = df / df.sum(axis=0)
    ###
    plt.figure(figsize=(8, 8))
    _ = plt.pcolor(norm_df)
    _ = plt.xticks(np.arange(0.5, len(df.columns), 1), df.columns, rotation=90)
    _ = plt.yticks(np.arange(0.5, len(df.index), 1), df.index)
    plt.xlabel("Predicted")
    plt.ylabel("Observed")
    plt.text(0.5, 1.05, f'Acc: {accuracy:.4f}', fontsize=12, ha='center', transform=plt.gca().transAxes)
    plt.savefig(f'{subset_path}Transfer_annotation_SCANVI_concordance_corrplot.pdf', bbox_inches='tight')
    plt.close()
    ###
    # save
    adata.obs.to_csv(f'{subset_path}Atlas_level_integration_{tmp_lineage}_metadata.txt', sep='\t')

In [ ]:
categories_order_TNK = ["CD4_Tn","CD4_Tcm",'CD4_Tsen',"CD4_T_CD26","CD4_CTL","CD4_Th17","CD4_Tfh-like",
                        "CD4_Tph-like","CD4_T_proliferative","Treg_naive","Treg_effector",
                        "CD8_Tn","CD8_Tcm",'CD8_Tem',"CD8_Temra","CD8_Trm","CD8_Tex_1",
                        "CD8_Tex_2","CD8_T_proliferative","NK_CD56_CD49a_Hi_CD103_Hi","NK_CD56_CD49a_Hi_CD103_Lo",
                        "NK_CD56_CD49a_Lo","NK_CD56_dim",'ILC3','MAIT', 'unknown']

In [ ]:
scANVI_integration(adata = adata_TNK, tmp_lineage = 'TNK')
scANVI_annotation_transfer(adata = adata_TNK, 
                           tmp_lineage = 'TNK', 
                           celltype_use = 'celltype', 
                           categories_order=categories_order_TNK)

## scVI with Pan-cancer CD4<sup>+</sup> T

In [ ]:
adata = sc.read("Reproducibility/Data/External_cohort/adata_concat_w_pancancer_CD4_T.h5ad")
adata = adata[adata.obs['cancerType'].isin(['CHOL'])==False]

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, 
                            n_top_genes=3000, 
                            flavor='seurat_v3',
                            layer="counts",
                            subset=True)

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    batch_key="libraryID")

model = scvi.model.SCVI(adata)
model.view_anndata_setup()

In [ ]:
model.train(max_epochs = 1000, early_stopping=True)
model.save("path_to_save", overwrite = True)

## BCG cohort

In [ ]:
adata = sc.read("Reproducibility/Data/External_cohort/adata_concat_w_npj.h5ad")

In [ ]:
def scANVI_integration(adata,tmp_lineage):
    sc.pp.highly_variable_genes(adata, 
                            n_top_genes=3000, 
                            flavor='seurat_v3',
                            layer="counts",
                            batch_key="study",
                            subset=True)
    ###
    adata_DOGMA = adata[adata.obs['study'].isin(['DOGMA'])==True].copy()
    adata_EXT = adata[adata.obs['study'].isin(['DOGMA'])==False].copy()
    ###
    ## Integration with scVI ##
    scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="batch_id")
    scvi_model = scvi.model.SCVI(adata, n_layers=2, n_latent=30)
    ###
    scvi_model.train(max_epochs=1000, early_stopping=True)
    ###
    subset_path = f'Reproducibility/Results/scANVI/BCG/{tmp_lineage}/'
    os.makedirs(subset_path, exist_ok = True)
    scvi_model_path = f'{subset_path}integration_batch_id/'
    scvi_model.save(scvi_model_path, overwrite=True)
    ###
    fig, ax = plt.subplots(1, 1)
    scvi_model.history["elbo_train"].plot(ax=ax, label="train")
    scvi_model.history["elbo_validation"].plot(ax=ax, label="validation")
    ax.set(title="Negative ELBO over training epochs")
    ax.legend()
    plt.savefig(f'{scvi_model_path}ELBO_curve_plot.pdf', bbox_inches='tight')
    plt.close()
    ###
    ## Load
    # scvi_model = scvi.model.SCVI.load(scvi_model_path, adata)
    SCVI_LATENT_KEY = "X_scVI"
    adata.obsm[SCVI_LATENT_KEY] = scvi_model.get_latent_representation()
    ###
    sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)
    sc.tl.umap(adata, min_dist=0.5)
    ###
    sc.set_figure_params(figsize=(3, 3))
    sc.set_figure_params(vector_friendly=True, dpi_save=100)
    sc.pl.umap(
        adata,
        color=["study",'batch_id','celltype'],
        frameon=False,
        ncols=4,
    #    size=0.3,
        wspace = 0.5,
    #    legend_loc="on data",
        legend_fontsize=4,
        show = False
    )
    plt.savefig(f'{scvi_model_path}Atlas_level_integration_UMAP.pdf', bbox_inches='tight')
    plt.close()

In [ ]:
def scANVI_annotation_transfer(adata,tmp_lineage,celltype_use,categories_order):
    subset_path = f'Reproducibility/Results/scANVI/BCG/{tmp_lineage}/'
    scvi_model_path = f'Reproducibility/Results/scANVI/BCG/{tmp_lineage}/integration_batch_id/'
    scvi_model = scvi.model.SCVI.load(scvi_model_path, adata)
    ## Transfer of annotations with scANVI ##
    SCANVI_CELLTYPE_KEY = celltype_use
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        scvi_model,
        adata=adata,
        unlabeled_category="unknown",
        labels_key=SCANVI_CELLTYPE_KEY,
    )
    ###
    scanvi_model.train(max_epochs=1000, early_stopping=True, 
                       n_samples_per_label=adata.obs[SCANVI_CELLTYPE_KEY].value_counts().min())
    ###
    scanvi_model_path = f'{subset_path}SCANVI_{SCANVI_CELLTYPE_KEY}/'
    scanvi_model.save(scanvi_model_path, overwrite=True)
    ###
    fig, ax = plt.subplots(1, 1)
    scanvi_model.history["elbo_train"].plot(ax=ax, label="train")
    scanvi_model.history["elbo_validation"].plot(ax=ax, label="validation")
    ax.set(title="Negative ELBO over training epochs")
    plt.savefig(f'{scanvi_model_path}ELBO_curve_plot.pdf', bbox_inches='tight')
    plt.close()
    ###
    ## Load
    # scanvi_model = scvi.model.SCANVI.load(scanvi_model_path, adata)
    ## scanvae = sca.models.SCANVI.load(scanvi_model, adata)
    ###
    SCANVI_LATENT_KEY = "X_scANVI"
    SCANVI_PREDICTION_KEY = f'predicted_{SCANVI_CELLTYPE_KEY}'
    ###
    adata.obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation(adata)
    adata.obs[SCANVI_PREDICTION_KEY] = scanvi_model.predict(adata)
    ###
    sc.pp.neighbors(adata, use_rep=SCANVI_LATENT_KEY)
    sc.tl.umap(adata, min_dist=0.5)
    ###
    categories_order2 = list(filter(lambda x: x != 'unknown', categories_order))
    ###
    adata.obs[SCANVI_CELLTYPE_KEY] = pd.Categorical(adata.obs[SCANVI_CELLTYPE_KEY], categories=categories_order, ordered=True)
    adata.obs[SCANVI_PREDICTION_KEY] = pd.Categorical(adata.obs[SCANVI_PREDICTION_KEY], categories=categories_order2, ordered=True)
    ###
    sc.set_figure_params(figsize=(3, 3))
    sc.set_figure_params(vector_friendly=True, dpi_save=100)
    sc.pl.umap(
        adata,
        color=["study",'batch_id','celltype',SCANVI_PREDICTION_KEY],
        frameon=False,
        ncols=4,
    #    size=0.3,
        wspace = 0.5,
    #    legend_loc="on data",
        legend_fontsize=4,
        show = False
    )
    plt.savefig(f'{subset_path}Transfer_annotation_SCANVI_UMAP.pdf', bbox_inches='tight')
    plt.close()
    ###
    # concordance
    tmp = adata[adata.obs['study'].isin(['DOGMA'])==True].copy()
    ###
    accuracy = np.mean(tmp.obs[SCANVI_PREDICTION_KEY].astype(str) == tmp.obs[SCANVI_CELLTYPE_KEY].astype(str))
    print("Acc: {}".format(accuracy))
    ###
    df = tmp.obs.groupby([SCANVI_CELLTYPE_KEY, SCANVI_PREDICTION_KEY]).size().unstack(fill_value=0)
    norm_df = df / df.sum(axis=0)
    ###
    plt.figure(figsize=(8, 8))
    _ = plt.pcolor(norm_df)
    _ = plt.xticks(np.arange(0.5, len(df.columns), 1), df.columns, rotation=90)
    _ = plt.yticks(np.arange(0.5, len(df.index), 1), df.index)
    plt.xlabel("Predicted")
    plt.ylabel("Observed")
    plt.text(0.5, 1.05, f'Acc: {accuracy:.4f}', fontsize=12, ha='center', transform=plt.gca().transAxes)
    plt.savefig(f'{subset_path}Transfer_annotation_SCANVI_concordance_corrplot.pdf', bbox_inches='tight')
    plt.close()
    ###
    # save
    adata.obs.to_csv(f'{subset_path}Atlas_level_integration_{tmp_lineage}_metadata.txt', sep='\t')

MSC

In [ ]:
adata_tmp = adata[adata.obs['lineage_concat'].isin(['MSC'])].copy()

condition_to_drop = (adata_tmp.obs['study'] == 'DOGMA') & (adata_tmp.obs['lineage'] != 'MSC')
adata_tmp = adata_tmp[~condition_to_drop].copy()

if adata_tmp.obs['celltype'].dtype.name == 'category':
    adata_tmp.obs['celltype'] = adata_tmp.obs['celltype'].cat.add_categories(['unknown'])

adata_tmp.obs.loc[adata_tmp.obs['study'] == 'npj', 'celltype'] = 'unknown'

categories_order_MSC = ["proCAF","iCAF_CD321",'iCAF_SLC14A1',"matCAF","matCAP",
                        'contCAP','vSMC', 'unknown']

scANVI_integration(adata = adata_tmp, tmp_lineage = 'MSC')
scANVI_annotation_transfer(adata = adata_tmp, 
                           tmp_lineage = 'MSC', 
                           celltype_use = 'celltype', 
                           categories_order=categories_order_MSC)


B

In [ ]:
adata_tmp = adata[adata.obs['lineage_concat'].isin(['B'])].copy()

condition_to_remove = (
    ((adata_tmp.obs['study'] == 'DOGMA') & (adata_tmp.obs['lineage'] != 'B')) | 
    ((adata_tmp.obs['study'] == 'DOGMA') & (adata_tmp.obs['celltype'] == 'Plasma'))
)

adata_tmp = adata_tmp[~condition_to_remove].copy()

if adata_tmp.obs['celltype'].dtype.name == 'category':
    adata_tmp.obs['celltype'] = adata_tmp.obs['celltype'].cat.add_categories(['unknown'])

adata_tmp.obs.loc[adata_tmp.obs['study'] == 'npj', 'celltype'] = 'unknown'

categories_order_B = ["B_naive","B_memory",'Atypical_B',"GC_B",'unknown']

scANVI_integration(adata = adata_tmp, tmp_lineage = 'B')
scANVI_annotation_transfer(adata = adata_tmp, 
                           tmp_lineage = 'B', 
                           celltype_use = 'celltype', 
                           categories_order=categories_order_B)

Myeloid

In [ ]:
adata_tmp = adata[adata.obs['lineage_concat'].isin(['Myeloid'])].copy()

condition_to_drop = (adata_tmp.obs['study'] == 'DOGMA') & (adata_tmp.obs['lineage'] != 'Myeloid')
adata_tmp = adata_tmp[~condition_to_drop].copy()

if adata_tmp.obs['celltype'].dtype.name == 'category':
    adata_tmp.obs['celltype'] = adata_tmp.obs['celltype'].cat.add_categories(['unknown'])

adata_tmp.obs.loc[adata_tmp.obs['study'] == 'npj', 'celltype'] = 'unknown'

categories_order_Myeloid = ['Mono','MDSC-like','TAM_TREM2','TAM_FOLR2','cDC1','cDC2',
                            'mregDC','preDC','pDC','unknown']

scANVI_integration(adata = adata_tmp, tmp_lineage = 'Myeloid')
scANVI_annotation_transfer(adata = adata_tmp, 
                           tmp_lineage = 'Myeloid', 
                           celltype_use = 'celltype', 
                           categories_order=categories_order_Myeloid)


TNK

In [ ]:
adata_tmp = adata[adata.obs['lineage_concat'].isin(['TNK'])].copy()

condition_to_drop = (adata_tmp.obs['study'] == 'DOGMA') & (adata_tmp.obs['lineage'].isin(['CD4_T','CD8_T_NK_ILC'])==False)
adata_tmp = adata_tmp[~condition_to_drop].copy()

if adata_tmp.obs['celltype'].dtype.name == 'category':
    adata_tmp.obs['celltype'] = adata_tmp.obs['celltype'].cat.add_categories(['unknown'])

adata_tmp.obs.loc[adata_tmp.obs['study'] == 'npj', 'celltype'] = 'unknown'

categories_order_TNK = ["CD4_Tn","CD4_Tcm",'CD4_Tsen',"CD4_T_CD26","CD4_CTL","CD4_Th17","CD4_Tfh-like",
                        "CD4_Tph-like","CD4_T_proliferative","Treg_naive","Treg_effector",
                        "CD8_Tn","CD8_Tcm",'CD8_Tem',"CD8_Temra","CD8_Trm","CD8_Tex_1",
                        "CD8_Tex_2","CD8_T_proliferative","NK_CD56_CD49a_Hi_CD103_Hi","NK_CD56_CD49a_Hi_CD103_Lo",
                        "NK_CD56_CD49a_Lo","NK_CD56_dim",'ILC3','MAIT', 'unknown']

scANVI_integration(adata = adata_tmp, tmp_lineage = 'TNK')
scANVI_annotation_transfer(adata = adata_tmp, 
                           tmp_lineage = 'TNK', 
                           celltype_use = 'celltype', 
                           categories_order=categories_order_TNK)

Data summarization

In [ ]:
model_dir = "Reproducibility/Results/scANVI/BCG"
meta_B = pd.read_csv(f"{model_dir}/B/Atlas_level_integration_B_metadata.txt", sep='\t', index_col=0)
meta_TNK = pd.read_csv(f"{model_dir}/TNK/Atlas_level_integration_TNK_metadata.txt", sep='\t', index_col=0)
meta_MSC = pd.read_csv(f"{model_dir}/MSC/Atlas_level_integration_MSC_metadata.txt", sep='\t', index_col=0)
meta_Myeloid = pd.read_csv(f"{model_dir}/Myeloid/Atlas_level_integration_Myeloid_metadata.txt", sep='\t', index_col=0)

In [ ]:
adata.obs['predicted_celltype'] = adata.obs['lineage_concat']
adata.obs['predicted_celltype'] = adata.obs.index.map(
    lambda k: meta_B.loc[k, 'predicted_celltype'] if k in meta_B.index else adata.obs['predicted_celltype'][k]
)
adata.obs['predicted_celltype'] = adata.obs.index.map(
    lambda k: meta_TNK.loc[k, 'predicted_celltype'] if k in meta_TNK.index else adata.obs['predicted_celltype'][k]
)
adata.obs['predicted_celltype'] = adata.obs.index.map(
    lambda k: meta_MSC.loc[k, 'predicted_celltype'] if k in meta_MSC.index else adata.obs['predicted_celltype'][k]
)
adata.obs['predicted_celltype'] = adata.obs.index.map(
    lambda k: meta_Myeloid.loc[k, 'predicted_celltype'] if k in meta_Myeloid.index else adata.obs['predicted_celltype'][k]
)

meta_npj= adata.obs[adata.obs['study']=='npj'].copy()
meta_npj.to_csv(f"{model_dir}/annotation_transfer_metadata.txt", sep='\t')

Epithelial signature score

In [ ]:
mask = (adata.obs['study'] == 'npj') & (adata.obs['lineage_concat'].isin(['Epithelial']))
adata_tmp = adata[mask].copy()

In [ ]:
sc.pp.normalize_total(adata_tmp)
sc.pp.log1p(adata_tmp)
genes = adata_tmp.var.index.values.tolist()

In [ ]:
data = {'pEMT_pancancer': ['ACTB', 'ACTG1', 'ADIRF', 'AKAP12', 'AKR1C1', 'ANXA1', 'ANXA2', 'ANXA3', 'ANXA5', 'ARL4C', 'ARPC1B', 
                    'BCAM', 'BPGM', 'C19orf33', 'CAPN2', 'CD151', 'CD24', 'CD44', 'CD59', 'CD68', 'CD9', 'CD99', 'CDA', 
                    'CDKN2A', 'CFL1', 'CHI3L1', 'CLIC1', 'COTL1', 'CRIP1', 'CSTB', 'CTGF', 'CTSB', 'DKK1', 'ECM1', 'EIF4EBP1', 
                    'EIF5A', 'EMP3', 'ENO1', 'F3', 'FLNA', 'FLNB', 'FSTL3', 'FTL', 'G0S2', 'G6PD', 'GAPDH', 'GJB6', 'GLO1', 
                    'GPI', 'GPX1', 'HMGA1', 'IGFBP2', 'IGFBP6', 'IGFBP7', 'INHBA', 'ITGA6', 'ITGB1', 'ITGB6', 'ITGB8', 
                    'ITM2B', 'KLK6', 'KRT17', 'KRT18', 'KRT19', 'KRT7', 'KRT8', 'KYNU', 'LAMA3', 'LAMB3', 'LAMC2', 'LGALS1', 
                    'LGALS3', 'LINC00152', 'MGST1', 'MIF', 'MMP1', 'MMP3', 'MT1E', 'MT2A', 'MYH9', 'MYL12B', 'MYL6', 'MYO1B', 
                    'OCIAD2', 'ODC1', 'PDLIM4', 'PDLIM7', 'PDPN', 'PFN1', 'PFN2', 'PGAM1', 'PKM', 'PLA2G16', 'PLAU', 'PLEK2',
                    'PLP2', 'PMEPA1', 'PODXL', 'PRKCDBP', 'PRSS23', 'PTRF', 'RBP1', 'RHOC', 'RHOD', 'S100A10', 'S100A11', 
                    'S100A13', 'S100A16', 'S100A2', 'S100A6', 'SERINC2', 'SERPINE1', 'SERPINE2', 'SH3BGRL3', 'SLC25A6', 
                    'SLC38A5', 'SMS', 'SPINK6', 'STEAP4', 'STOM', 'SULT1E1', 'TAGLN', 'TAGLN2', 'TGFBI', 'THBS1', 'TIMP1', 
                    'TIMP3', 'TMSB10', 'TMSB4X', 'TNC', 'TNFRSF12A', 'TNFRSF21', 'TNFRSF6B', 'TPM1', 'TPM2', 'TPM4', 'TRIM29', 
                    'TSHZ2', 'TXN', 'UACA', 'VIM', 'WDR1', 'YBX1', 'ZBED2'], 
  'Interferon_pancancer': ['ACTN1', 'APOL1', 'APOL2', 'APOL3', 'APOL6', 'B2M', 'BST2', 'C19orf66', 'C1R', 'C1S', 'C3', 'C5orf56', 
                 'CCL2', 'CCL27', 'CCL5', 'CD74', 'CFB', 'CFH', 'CMPK2', 'CX3CL1', 'CXCL10', 'CXCL11', 'CXCL9', 'DDX58', 
                 'DDX60', 'DDX60L', 'DYNLT1', 'EIF2AK2', 'EPSTI1', 'ETV7', 'FXYD5', 'GBP1', 'GBP2', 'GBP3', 'GBP4', 'GBP5', 
                 'GLUL', 'HAPLN3', 'HELZ2', 'HERC5', 'HERC6', 'HLA-A', 'HLA-B', 'HLA-C', 'HLA-DMA', 'HLA-DMB', 'HLA-DPA1', 
                 'HLA-DPB1', 'HLA-DQA1', 'HLA-DQA2', 'HLA-DQB1', 'HLA-DRA', 'HLA-DRB1', 'HLA-DRB5', 'HLA-E', 'HLA-F', 'HSPB8', 
                 'ICAM1', 'IDO1', 'IFI16', 'IFI27', 'IFI35', 'IFI44', 'IFI44L', 'IFI6', 'IFIH1', 'IFIT1', 'IFIT2', 'IFIT3', 
                 'IFITM1', 'IFITM2', 'IFITM3', 'IL32', 'IRF1', 'IRF7', 'ISG15', 'ISG20', 'KLHDC7B', 'LAMP3', 'LAP3', 
                 'LGALS3BP', 'LGALS9', 'LY6E', 'MDK', 'MSRB1', 'MX1', 'MX2', 'NNMT', 'NT5C3A', 'NUB1', 'OAS1', 'OAS2', 'OAS3', 
                 'OASL', 'PARP14', 'PARP9', 'PDZK1IP1', 'PLSCR1', 'PSMB10', 'PSMB8', 'PSMB9', 'PSME1', 'PSME2', 'RARRES3', 
                 'RNF213', 'RSAD2', 'RTP4', 'SAA1', 'SAMD9', 'SAMD9L', 'SAMHD1', 'SERPING1', 'SERPINH1', 'SLFN5', 'SOD2', 
                 'SP100', 'SP110', 'SPP1', 'SQRDL', 'STAT1', 'TAP1', 'TAP2.1', 'TAPBP', 'TGM2', 'TNFSF10', 'TNFSF13B', 
                 'TRIM22', 'TYMP', 'UBD', 'UBE2L6', 'USP18', 'VCAM1', 'WARS', 'XAF1', 'ZC3HAV1', 'ZNFX1']}


In [ ]:
sig_names = list(data)
for tmp_sig in sig_names:
  sig_score_names = tmp_sig + '_score'
  gene_list_tmp = data[tmp_sig] 
  gene_list_tmp_filt = [i for i in gene_list_tmp if i in genes]
  sc.tl.score_genes(adata_tmp, gene_list_tmp_filt, ctrl_size=50, gene_pool=None, n_bins=25, 
                    score_name= sig_score_names,
                    random_state=1234, copy=False, use_raw=None)

In [ ]:
sig_score_names = [name + '_score' for name in sig_names]
sig_df = adata_tmp.obs.filter(sig_score_names)

tmp_path = f"{model_dir}/External_dataset_malignant_signature_score_scanpy.txt"
sig_df.to_csv(tmp_path, sep='\t')